#### Import Libraries

In [1]:
#### Import Libraries

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

import joblib

#### Loading the Dataset

In [2]:
df = pd.read_csv("../data/raw/defects_data.csv")

print("Dataset Shape:", df.shape)
print("\nDataset Columns:")
print(df.columns.tolist())

display(df.head(10))

Dataset Shape: (1000, 8)

Dataset Columns:
['defect_id', 'product_id', 'defect_type', 'defect_date', 'defect_location', 'severity', 'inspection_method', 'repair_cost']


,defect_id,product_id,defect_type,defect_date,defect_location,severity,inspection_method,repair_cost
0,1,15,Structural,6/6/2024,Component,Minor,Visual Inspection,245.47
1,2,6,Functional,4/26/2024,Component,Minor,Visual Inspection,26.87
2,3,84,Structural,2/15/2024,Internal,Minor,Automated Testing,835.81
3,4,10,Functional,3/28/2024,Internal,Critical,Automated Testing,444.47
4,5,14,Cosmetic,4/26/2024,Component,Minor,Manual Testing,823.64
5,6,17,Functional,5/11/2024,Internal,Moderate,Visual Inspection,788.11
6,7,85,Cosmetic,5/23/2024,Internal,Critical,Manual Testing,33.68
7,8,90,Structural,1/15/2024,Internal,Moderate,Manual Testing,65.56
8,9,30,Structural,1/26/2024,Component,Critical,Manual Testing,848.61
9,10,20,Structural,6/5/2024,Component,Critical,Visual Inspection,478.48


#### Validating Data Types and Missing Values

In [3]:
print("Dataset Shape:", df.shape)

print("\nData Types:")
display(df.dtypes.to_frame(name="Data Type"))

print("\nMissing Values:")
display(df.isnull().sum().to_frame(name="Missing Values"))

print("\nDuplicate Records:", df.duplicated().sum())

print("\nUnique Values per Column:")
display(df.nunique().to_frame(name="Unique Values"))

Dataset Shape: (1000, 8)

Data Types:


,Data Type
defect_id,int64
product_id,int64
defect_type,str
defect_date,str
defect_location,str
severity,str
inspection_method,str
repair_cost,float64



Missing Values:


,Missing Values
defect_id,0
product_id,0
defect_type,0
defect_date,0
defect_location,0
severity,0
inspection_method,0
repair_cost,0



Duplicate Records: 0

Unique Values per Column:


,Unique Values
defect_id,1000
product_id,100
defect_type,3
defect_date,182
defect_location,3
severity,3
inspection_method,3
repair_cost,996


Key observations

- defect_id: 1,000 unique values across 1,000 records. This confirms its role as a record identifier.
- product_id: 100 unique products. We should treat this as an identifier, not a continuous numerical measurement.
- defect_date: 182 distinct dates. This provides an opportunity for temporal feature engineering.
- severity: Three distinct classes, confirming our multiclass classification target.
- repair_cost: 996 distinct values. This is a continuous measurement, but its availability at prediction time remains uncertain.

#### Selecting Target and Predictor Variables

In [4]:
#### Selecting Target and Predictor Variables

target_column = "severity"

excluded_columns = [
    "defect_id",
    "repair_cost"
]

X = df.drop(columns=[target_column] + excluded_columns)
y = df[target_column].copy()

print("Predictor Dataset Shape:", X.shape)
print("Target Dataset Shape:", y.shape)

print("\nSelected Predictor Variables:")
print(X.columns.tolist())

print("\nTarget Class Distribution:")
display(y.value_counts().to_frame(name="Count"))

display(X.head())

Predictor Dataset Shape: (1000, 5)
Target Dataset Shape: (1000,)

Selected Predictor Variables:
['product_id', 'defect_type', 'defect_date', 'defect_location', 'inspection_method']

Target Class Distribution:


,Count
severity,
Minor,358
Critical,333
Moderate,309


,product_id,defect_type,defect_date,defect_location,inspection_method
0,15,Structural,6/6/2024,Component,Visual Inspection
1,6,Functional,4/26/2024,Component,Visual Inspection
2,84,Structural,2/15/2024,Internal,Automated Testing
3,10,Functional,3/28/2024,Internal,Automated Testing
4,14,Cosmetic,4/26/2024,Component,Manual Testing


#### Observations on Feature and Target Selection

- The dataset contains **1,000 records**, with 5 selected predictor variables and 1 target variable (`severity`).
- The target variable contains three classes: **Minor (358), Critical (333), and Moderate (309)**, representing a relatively balanced multiclass classification problem.
- `defect_id` was excluded because it serves as a unique record identifier and does not represent a meaningful defect characteristic.
- `repair_cost` was excluded from the predictive features to avoid potential target leakage, assuming that repair costs are determined after defect severity is assessed.
- `product_id` has been retained temporarily for further evaluation. Since it represents product identity rather than a continuous measurement, it should not be treated as an ordinary numerical feature.
- `defect_date` will require conversion to a datetime format before temporal feature engineering.

#### Validating and Converting the Date Variable

In [5]:
#### Converting and Validating the Date Variable

X["defect_date"] = pd.to_datetime(
    X["defect_date"],
    format="%m/%d/%Y",
    errors="coerce"
)

print("Date Variable Data Type:")
print(X["defect_date"].dtype)

print("\nDate Range:")
print("Earliest Date:", X["defect_date"].min())
print("Latest Date:", X["defect_date"].max())

print("\nInvalid or Missing Dates:")
print(X["defect_date"].isna().sum())

print("\nUnique Dates:")
print(X["defect_date"].nunique())

display(X.head())

Date Variable Data Type:
datetime64[us]

Date Range:
Earliest Date: 2024-01-01 00:00:00
Latest Date: 2024-06-30 00:00:00

Invalid or Missing Dates:
0

Unique Dates:
182


,product_id,defect_type,defect_date,defect_location,inspection_method
0,15,Structural,2024-06-06,Component,Visual Inspection
1,6,Functional,2024-04-26,Component,Visual Inspection
2,84,Structural,2024-02-15,Internal,Automated Testing
3,10,Functional,2024-03-28,Internal,Automated Testing
4,14,Cosmetic,2024-04-26,Component,Manual Testing


#### Observations on Date Conversion

- The `defect_date` variable has been converted from its original string representation into a datetime format.
- The displayed records confirm that dates are represented consistently in the `YYYY-MM-DD` format.
- Converting the date variable enables the extraction of temporal features during the feature engineering stage.
- The date range and number of invalid or missing dates must be verified before proceeding with temporal transformations.

#### Evaluating Product ID

In [6]:
product_counts = X["product_id"].value_counts()

print("Total Unique Products:", X["product_id"].nunique())

print("\nProduct Frequency Statistics:")
display(product_counts.describe().to_frame(name="Value"))

print("\nProducts Appearing Only Once:")
print((product_counts == 1).sum())

print("\nProducts Appearing Multiple Times:")
print((product_counts > 1).sum())

print("\nTop 10 Most Frequent Products:")
display(
    product_counts.head(10).to_frame(name="Defect Count")
)

print("\nDate Validation:")
print("Earliest Date:", X["defect_date"].min())
print("Latest Date:", X["defect_date"].max())
print("Invalid or Missing Dates:", X["defect_date"].isna().sum())

Total Unique Products: 100

Product Frequency Statistics:


,Value
count,100.000000
mean,10.000000
std,3.601347
min,3.000000
25%,7.000000
50%,10.000000
75%,12.000000
max,20.000000



Products Appearing Only Once:
0

Products Appearing Multiple Times:
100

Top 10 Most Frequent Products:


,Defect Count
product_id,
63,20
81,20
97,19
56,18
10,16
92,16
4,16
6,15
45,15



Date Validation:
Earliest Date: 2024-01-01 00:00:00
Latest Date: 2024-06-30 00:00:00
Invalid or Missing Dates: 0


#### Observations on Product ID and Date Validation

- The dataset contains **100 unique products across 1,000 defect records**, with an average of 10 defect records per product.
- Every product appears multiple times, with frequencies ranging from 3 to 20 records. No products occur only once.
- The median product frequency is 10, indicating that most products have relatively limited historical observations.
- Although `product_id` may contain product-specific information, it represents a categorical identifier rather than a continuous numerical variable.
- The dataset covers **January 1, 2024, through June 30, 2024**, providing six months of historical defect records.
- No invalid or missing dates were identified, confirming that `defect_date` can be used for temporal feature engineering.

#### Feature Selection and Exclusion Rationale

Feature selection was performed to retain variables relevant to defect severity prediction while minimizing potential data leakage, unnecessary model complexity, and reliance on non-informative identifiers.

The following variables were excluded from the predictor dataset:

**1. `defect_id` — Unique Record Identifier**

- Contains 1,000 unique values across 1,000 records.
- Functions as a record identifier rather than a meaningful manufacturing defect characteristic.
- Including this variable could introduce artificial relationships based on record numbering rather than genuine severity patterns.

**Decision:** Excluded from model training.

**2. `product_id` — High-Cardinality Product Identifier**

- Contains 100 unique product identifiers, with each product appearing between 3 and 20 times.
- Represents product identity rather than a continuous numerical measurement.
- Treating the identifier as numerical would introduce an artificial ordering between products.
- One-hot encoding could introduce up to 100 additional features, increasing model complexity relative to the dataset size.
- Limited observations per product may make product-specific severity patterns difficult to estimate reliably.

**Decision:** Excluded from the primary modeling features. Its predictive value may be reassessed in future experiments.

**3. `repair_cost` — Potential Target Leakage**

- Represents the financial cost associated with repairing a manufacturing defect.
- The prediction objective is to classify defect severity using information available at inspection time.
- If repair costs are determined after severity assessment, including this variable would introduce information unavailable when predictions are required.
- Such leakage could produce misleadingly optimistic model performance and reduce real-world applicability.

**Decision:** Excluded from model training as a precaution against target leakage, under the assumed prediction-time workflow.

**Final Predictor Variables**

The following variables were retained:

- `defect_type` — Describes the nature of the identified defect.
- `defect_location` — Identifies where the defect occurred.
- `inspection_method` — Indicates the method used to inspect the defect.
- `defect_date` — Provides temporal information for subsequent feature engineering.

**Target Variable:** `severity` (Minor, Moderate, Critical).

The resulting predictor dataset contains **4 original features across 1,000 records**, prior to temporal feature engineering and categorical encoding.

#### Finalizing Predictor Variables

In [7]:
X = X.drop(columns=["product_id"])

print("Final Predictor Dataset Shape:", X.shape)
print("Target Dataset Shape:", y.shape)

print("\nFinal Predictor Variables:")
print(X.columns.tolist())

print("\nPredictor Data Types:")
display(X.dtypes.to_frame(name="Data Type"))

print("\nMissing Values:")
display(X.isnull().sum().to_frame(name="Missing Values"))

display(X.head())

Final Predictor Dataset Shape: (1000, 4)
Target Dataset Shape: (1000,)

Final Predictor Variables:
['defect_type', 'defect_date', 'defect_location', 'inspection_method']

Predictor Data Types:


,Data Type
defect_type,str
defect_date,datetime64[us]
defect_location,str
inspection_method,str



Missing Values:


,Missing Values
defect_type,0
defect_date,0
defect_location,0
inspection_method,0


,defect_type,defect_date,defect_location,inspection_method
0,Structural,2024-06-06,Component,Visual Inspection
1,Functional,2024-04-26,Component,Visual Inspection
2,Structural,2024-02-15,Internal,Automated Testing
3,Functional,2024-03-28,Internal,Automated Testing
4,Cosmetic,2024-04-26,Component,Manual Testing


#### Splitting the Dataset

In [8]:
#### Splitting the Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

print("\nTraining Target Distribution:")
display(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame(name="Percentage (%)")
)

print("\nTesting Target Distribution:")
display(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame(name="Percentage (%)")
)

Training Features Shape: (800, 4)
Testing Features Shape: (200, 4)

Training Target Shape: (800,)
Testing Target Shape: (200,)

Training Target Distribution:


,Percentage (%)
severity,
Minor,35.75
Critical,33.38
Moderate,30.88



Testing Target Distribution:


,Percentage (%)
severity,
Minor,36.0
Critical,33.0
Moderate,31.0


#### Observations on Train/Test Split

- The dataset was divided into **800 training records (80%)** and **200 testing records (20%)**, with 4 original predictor variables.
- Stratified sampling was applied to preserve the distribution of the three severity classes across both datasets.
- The training dataset contains **35.75% Minor, 33.38% Critical, and 30.88% Moderate** defects.
- The testing dataset contains **36.00% Minor, 33.00% Critical, and 31.00% Moderate** defects.
- The closely aligned class proportions confirm that stratification successfully maintained a representative severity distribution.
- A fixed random state of `42` ensures reproducibility, while preprocessing transformations will be fitted exclusively on the training data to prevent data leakage.

#### Identifying Feature Types

In [9]:
#### Identifying Feature Types

categorical_features = [
    "defect_type",
    "defect_location",
    "inspection_method"
]

datetime_features = [
    "defect_date"
]

print("Categorical Features:")
print(categorical_features)

print("\nDatetime Features:")
print(datetime_features)

print("\nTraining Feature Data Types:")
display(X_train.dtypes.to_frame(name="Data Type"))

print("\nTesting Feature Data Types:")
display(X_test.dtypes.to_frame(name="Data Type"))

print("\nMissing Values in Training Data:")
display(X_train.isnull().sum().to_frame(name="Missing Values"))

print("\nMissing Values in Testing Data:")
display(X_test.isnull().sum().to_frame(name="Missing Values"))

Categorical Features:
['defect_type', 'defect_location', 'inspection_method']

Datetime Features:
['defect_date']

Training Feature Data Types:


,Data Type
defect_type,str
defect_date,datetime64[us]
defect_location,str
inspection_method,str



Testing Feature Data Types:


,Data Type
defect_type,str
defect_date,datetime64[us]
defect_location,str
inspection_method,str



Missing Values in Training Data:


,Missing Values
defect_type,0
defect_date,0
defect_location,0
inspection_method,0



Missing Values in Testing Data:


,Missing Values
defect_type,0
defect_date,0
defect_location,0
inspection_method,0


#### Observations on Feature Types and Data Validation

- Both training and testing datasets contain **4 predictor variables**, consisting of 3 categorical features and 1 datetime feature.
- The categorical variables are `defect_type`, `defect_location`, and `inspection_method`, which will require encoding before model training.
- The `defect_date` variable has been successfully converted to a datetime data type and is ready for temporal feature engineering.
- The feature data types are consistent across the training and testing datasets.
- No missing values were identified in either dataset, eliminating the need for missing-value imputation at this stage.
- Categorical encoding and other fitted transformations will be performed using the training data only to prevent data leakage.

#### Saving Train and Test Datasets


In [10]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Saving training and testing features
X_train.to_parquet(processed_dir / "X_train_raw.parquet")
X_test.to_parquet(processed_dir / "X_test_raw.parquet")

# Saving training and testing targets
y_train.to_frame(name="severity").to_parquet(
    processed_dir / "y_train.parquet"
)

y_test.to_frame(name="severity").to_parquet(
    processed_dir / "y_test.parquet"
)

print("Train and test datasets saved successfully.")

print("\nSaved Files:")
for file in sorted(processed_dir.glob("*.parquet")):
    print(file.name)

Train and test datasets saved successfully.

Saved Files:
X_test_raw.parquet
X_train_raw.parquet
y_test.parquet
y_train.parquet


#### Observations on Data Preparation

- The original dataset contained **1,000 records and 8 columns**, with `severity` selected as the target for multiclass classification.
- Three variables were excluded from the predictor dataset: `defect_id` (unique identifier), `product_id` (high-cardinality product identifier), and `repair_cost` (potential target leakage).
- The final predictor dataset contains **4 original features**: `defect_type`, `defect_location`, `inspection_method`, and `defect_date`.
- The `defect_date` variable was successfully converted to datetime format, with no invalid or missing dates identified.
- An **80:20 stratified train/test split** produced 800 training records and 200 testing records, preserving the proportions of Minor, Moderate, and Critical defects.
- No missing values were identified in either the training or testing features.
- The training and testing datasets were saved in **Parquet format** to preserve their data types and support reproducible processing in subsequent notebooks.
- Categorical encoding and temporal feature engineering have been deferred to `04_feature_engineering.ipynb`, ensuring that fitted preprocessing transformations are learned from training data only.